# Lab3.3 — DMA loopback: PS → PL → PS

În acest task realizăm primul transfer complet de date între memoria DDR din PS și un flux hardware din PL, apoi înapoi în DDR.


## Obiective

- alocarea bufferelor de intrare și ieșire;
- inițierea canalelor DMA;
- transferul unui vector PS → PL → PS;
- verificarea integrității datelor;
- utilizarea `wait()` pentru sincronizarea cu hardware-ul.


## 1. Pregătirea Overlay-ului


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

BITSTREAM_DIR = Path("bitstream")
BITSTREAM_DIR.mkdir(exist_ok=True)

BASE_URL = "https://raw.githubusercontent.com/onigas/PYNQ_Workshop/master/Session_4/bitstream"
FILES = ("dma_tutorial.bit", "dma_tutorial.hwh")

for filename in FILES:
    local_path = BITSTREAM_DIR / filename
    if not local_path.exists():
        print(f"Downloading {filename} ...")
        try:
            urlretrieve(f"{BASE_URL}/{filename}", local_path)
        except Exception as e:
            raise RuntimeError(
                f"Nu s-a putut descărca {filename}. "
                "Copiați manual fișierele dma_tutorial.bit și dma_tutorial.hwh "
                "în directorul Lab3/bitstream."
            ) from e

print("DMA overlay files are ready.")

In [ ]:
from pynq import Overlay, allocate
import numpy as np

overlay = Overlay("bitstream/dma_tutorial.bit")
dma_send = overlay.axi_dma_from_ps_to_pl
dma_recv = overlay.axi_dma_from_pl_to_ps

print("DMA initialized.")

## 2. Bufferele

Creăm un vector de intrare și un vector de ieșire de aceeași dimensiune.


In [ ]:
N = 100

input_buffer = allocate(shape=(N,), dtype=np.uint32)
output_buffer = allocate(shape=(N,), dtype=np.uint32)

input_buffer[:] = np.arange(N, dtype=np.uint32) + 0xCAFE0000
output_buffer[:] = 0

print("Input :", [hex(int(x)) for x in input_buffer[:8]])
print("Output:", [hex(int(x)) for x in output_buffer[:8]])

## 3. Transferul DMA

Pornim mai întâi canalul de recepție, apoi canalul de transmisie. La final așteptăm terminarea ambelor transferuri.


In [ ]:
if hasattr(input_buffer, "flush"):
    input_buffer.flush()

dma_recv.recvchannel.transfer(output_buffer)
dma_send.sendchannel.transfer(input_buffer)

dma_send.sendchannel.wait()
dma_recv.recvchannel.wait()

if hasattr(output_buffer, "invalidate"):
    output_buffer.invalidate()

print("DMA transfer completed.")

## 4. Verificarea rezultatului


In [ ]:
print("Input :", [hex(int(x)) for x in input_buffer[:8]])
print("Output:", [hex(int(x)) for x in output_buffer[:8]])

equal = np.array_equal(input_buffer, output_buffer)
print("Buffers are equal:", equal)

Dacă rezultatul este corect, diferența dintre cele două buffere trebuie să fie zero pentru toate elementele.


In [ ]:
differences = np.nonzero(input_buffer != output_buffer)[0]
print("Number of different elements:", len(differences))

## 5. Al doilea transfer

Modificăm datele de intrare și repetăm transferul pentru a demonstra că nu observăm doar date rămase din execuția anterioară.


In [ ]:
input_buffer[:] = np.arange(N, dtype=np.uint32) * 3 + 7
output_buffer[:] = 0

if hasattr(input_buffer, "flush"):
    input_buffer.flush()

dma_recv.recvchannel.transfer(output_buffer)
dma_send.sendchannel.transfer(input_buffer)
dma_send.sendchannel.wait()
dma_recv.recvchannel.wait()

if hasattr(output_buffer, "invalidate"):
    output_buffer.invalidate()

print("Buffers are equal:", np.array_equal(input_buffer, output_buffer))

## 6. Eliberarea memoriei


In [ ]:
del input_buffer, output_buffer
print("Buffers released.")

## 7. Exercițiu

Repetați experimentul cu **1024 elemente `uint32`** și inițializați bufferul cu pătratul indicelui: `0, 1, 4, 9, ...`. Verificați automat integritatea datelor după transfer.


In [ ]:
# Scrieți aici soluția


## 8. Întrebări

1. De ce sunt necesare două buffere?
2. De ce pornim recepția înaintea transmisiei în acest exemplu?
3. Ce garantează apelul `wait()`?
4. Ce demonstrează `np.array_equal()` în acest experiment și ce nu demonstrează despre performanță?
